# Estimation Methods Comparison: Copula vs KSG

**Circulatory Fidelity v1.1: Estimation Method Comparison**

This notebook compares the **recommended copula-based estimation** with the **KSG k-nearest neighbor** approach.

---

## Key Finding: Copula is the Unified Approach

The copula method is recommended for **all applications** because:
- It is **exact for Gaussian data** (returns |ρ| with <0.001 difference from Pearson)
- It provides **conservative estimates for non-Gaussian data**
- It enables a **unified workflow** without distributional verification

KSG is useful for:
- Validating copula estimates
- Detecting non-monotonic dependencies that copula misses


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import rankdata, norm
from scipy.special import digamma
from scipy.spatial import cKDTree
from typing import Tuple

plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 11
np.random.seed(42)

## Estimation Functions

In [ ]:
def ic_copula(x: np.ndarray, y: np.ndarray) -> Tuple[float, float]:
    """
    Copula-based IC estimation (RECOMMENDED).
    
    Algorithm:
    1. Rank-transform to uniform marginals
    2. Apply inverse normal CDF (probit)
    3. Compute Pearson correlation
    4. IC = |ρ|
    
    Returns: (ic, standard_error)
    """
    x = np.asarray(x).flatten()
    y = np.asarray(y).flatten()
    n = len(x)
    
    # Rank transform
    u = (rankdata(x) - 0.5) / n
    v = (rankdata(y) - 0.5) / n
    
    # Probit transform
    z_x = norm.ppf(u)
    z_y = norm.ppf(v)
    
    # Pearson correlation
    rho = np.corrcoef(z_x, z_y)[0, 1]
    ic = np.abs(rho)
    
    # Fisher transform SE
    se = 1.0 / np.sqrt(n - 3) if n > 3 else np.nan
    
    return ic, se


def mi_ksg(X: np.ndarray, Y: np.ndarray, k: int = 5) -> float:
    """
    KSG mutual information estimator.
    
    Note: This method has 30-45% negative bias for small samples.
    Use copula-based estimation as the primary method.
    """
    X = np.atleast_2d(X).T if X.ndim == 1 else X
    Y = np.atleast_2d(Y).T if Y.ndim == 1 else Y
    
    n = X.shape[0]
    XY = np.hstack([X, Y])
    
    tree_xy = cKDTree(XY)
    tree_x = cKDTree(X)
    tree_y = cKDTree(Y)
    
    distances, _ = tree_xy.query(XY, k=k+1, p=float('inf'))
    eps = distances[:, -1]
    eps = np.maximum(eps, 1e-10)
    
    n_x = np.array([len(tree_x.query_ball_point(X[i], r=eps[i], p=float('inf'))) - 1 
                   for i in range(n)])
    n_y = np.array([len(tree_y.query_ball_point(Y[i], r=eps[i], p=float('inf'))) - 1 
                   for i in range(n)])
    
    mi = digamma(k) + digamma(n) - np.mean(digamma(n_x + 1) + digamma(n_y + 1))
    return max(0, mi)


def ic_ksg(x: np.ndarray, y: np.ndarray, k: int = 5) -> Tuple[float, float]:
    """
    KSG-based IC estimation.
    
    IC = sqrt(1 - exp(-2*MI))
    
    Note: Has substantial negative bias. Use copula method as primary.
    """
    mi = mi_ksg(x, y, k=k)
    ic = np.sqrt(1 - np.exp(-2 * mi))
    
    # Bootstrap SE (expensive)
    n = len(x)
    boot_ic = []
    for _ in range(50):
        idx = np.random.choice(n, size=n, replace=True)
        mi_boot = mi_ksg(x[idx], y[idx], k=k)
        boot_ic.append(np.sqrt(1 - np.exp(-2 * max(0, mi_boot))))
    se = np.std(boot_ic)
    
    return ic, se

## Comparison: Gaussian Data

In [ ]:
def compare_estimators_gaussian(true_rho: float, n: int = 500, n_trials: int = 100):
    """Compare copula vs KSG on Gaussian data with known correlation."""
    true_ic = abs(true_rho)
    
    copula_estimates = []
    ksg_estimates = []
    
    for _ in range(n_trials):
        # Generate correlated Gaussian
        cov = [[1, true_rho], [true_rho, 1]]
        xy = np.random.multivariate_normal([0, 0], cov, n)
        x, y = xy[:, 0], xy[:, 1]
        
        ic_c, _ = ic_copula(x, y)
        ic_k, _ = ic_ksg(x, y, k=5)
        
        copula_estimates.append(ic_c)
        ksg_estimates.append(ic_k)
    
    return {
        'true_ic': true_ic,
        'copula_mean': np.mean(copula_estimates),
        'copula_std': np.std(copula_estimates),
        'copula_bias': np.mean(copula_estimates) - true_ic,
        'ksg_mean': np.mean(ksg_estimates),
        'ksg_std': np.std(ksg_estimates),
        'ksg_bias': np.mean(ksg_estimates) - true_ic,
    }

# Test across correlation values
rho_values = [0.2, 0.4, 0.6, 0.8]
results = [compare_estimators_gaussian(rho, n=300, n_trials=50) for rho in rho_values]
df_compare = pd.DataFrame(results)

print("Comparison: Copula vs KSG on Gaussian Data (n=300, 50 trials each)")
print(df_compare[['true_ic', 'copula_mean', 'copula_bias', 'ksg_mean', 'ksg_bias']].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel A: Estimates vs True
ax = axes[0]
ax.plot(df_compare['true_ic'], df_compare['true_ic'], 'k--', label='Perfect')
ax.errorbar(df_compare['true_ic'], df_compare['copula_mean'], 
           yerr=df_compare['copula_std'], fmt='o-', label='Copula', capsize=3)
ax.errorbar(df_compare['true_ic'] + 0.02, df_compare['ksg_mean'], 
           yerr=df_compare['ksg_std'], fmt='s-', label='KSG', capsize=3)
ax.set_xlabel('True IC')
ax.set_ylabel('Estimated IC')
ax.set_title('(A) Estimates vs True Value')
ax.legend()

# Panel B: Bias
ax = axes[1]
x = np.arange(len(rho_values))
width = 0.35
ax.bar(x - width/2, df_compare['copula_bias'], width, label='Copula bias', color='blue', alpha=0.7)
ax.bar(x + width/2, df_compare['ksg_bias'], width, label='KSG bias', color='orange', alpha=0.7)
ax.axhline(0, color='black', linestyle='--')
ax.set_xticks(x)
ax.set_xticklabels([f'ρ={r}' for r in rho_values])
ax.set_ylabel('Bias (Estimate - True)')
ax.set_title('(B) Estimation Bias')
ax.legend()

plt.tight_layout()
plt.show()

print(f"\nKSG shows {100*np.mean(np.abs(df_compare['ksg_bias'])):.1f}% average bias vs "
      f"{100*np.mean(np.abs(df_compare['copula_bias'])):.1f}% for copula")

## Non-Gaussian Example: Where KSG May Help

The copula method assumes **monotonic** dependence. For non-monotonic relationships (e.g., quadratic), KSG can detect dependence that copula misses.

In [ ]:
# Non-monotonic relationship: Y = X^2 + noise
n = 500
x = np.random.uniform(-2, 2, n)
y = x**2 + np.random.normal(0, 0.5, n)

ic_c, se_c = ic_copula(x, y)
ic_k, se_k = ic_ksg(x, y)

print("Non-monotonic relationship: Y = X² + noise")
print(f"  Copula IC: {ic_c:.3f} ± {se_c:.3f}")
print(f"  KSG IC:    {ic_k:.3f} ± {se_k:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].scatter(x, y, alpha=0.3, s=10)
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')
axes[0].set_title('Non-monotonic: Y = X² + noise')

# Compare estimates
methods = ['Copula', 'KSG']
estimates = [ic_c, ic_k]
colors = ['blue', 'orange']
axes[1].bar(methods, estimates, color=colors, alpha=0.7)
axes[1].set_ylabel('IC Estimate')
axes[1].set_title('IC Estimates\n(KSG detects non-monotonic dependence)')

plt.tight_layout()
plt.show()

## Two-Stage Protocol

The manuscript recommends a **two-stage protocol**:

1. **Stage 1**: Compute pairwise IC using copula estimation
   - If IC > threshold → coupling detected, assess MFVI risk
   - If IC ≈ 0 → proceed to Stage 2

2. **Stage 2**: If pairwise IC ≈ 0, check for synergistic (XOR-type) coupling
   - Compute interaction IC using higher-order analysis
   - If interaction IC > 0 → synergistic coupling detected

## Recommendations

| Scenario | Recommended Method |
|----------|-------------------|
| **All standard use** | **Copula** (unified workflow) |
| Gaussian data | Copula (exact) or Pearson |
| Non-Gaussian data | Copula (conservative) |
| Validation | KSG (for comparison) |
| Non-monotonic dependence suspected | KSG or `check_nonmonotonic_dependence()` |

### Why Copula is the Unified Approach

1. **Exact for Gaussians**: Returns |ρ| with negligible numerical difference (<0.001)
2. **Conservative for non-Gaussians**: Provides lower bound on true IC
3. **No verification needed**: Works correctly without checking distributional assumptions
4. **Closed-form SE**: Fisher transform provides analytic standard errors
5. **Computational efficiency**: Faster than KSG for large samples

### When to Use KSG

- When validating copula estimates on new model classes
- When non-monotonic dependence is suspected (copula returns ~0 for Y=X²)
- As a robustness check in high-stakes applications
